In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, substring, trim
from pyspark.sql.types import IntegerType

# Lire depuis Bronze
df = spark.read.format("delta").load(
    "abfss://WS_Projet_Ryma@onelake.dfs.fabric.microsoft.com/LH_Bronze.Lakehouse/Tables/dbo/commandes_brutes"
)

display(df)
df.printSchema()

# Nettoyage
df_clean = (df
    .dropDuplicates(["commande_id", "ligne"])
    .filter((col("client_id").isNotNull()) & (trim(col("client_id")) != ""))
    .filter((col("produit_id").isNotNull()) & (trim(col("produit_id")) != ""))
    .filter(col("quantite").cast("int") > 0)
    .filter(col("date_commande").rlike("^20(24|25)-\\d{2}-\\d{2}"))
    .withColumn("quantite", col("quantite").cast(IntegerType()))
    .withColumn("prix_unitaire", col("prix_unitaire").cast(IntegerType()))
    .withColumn("montant_brut", col("montant_brut").cast(IntegerType()))
    .withColumn("remise_pct", col("remise_pct").cast(IntegerType()))
    .withColumn(
        "montant_net",
        (col("montant_brut") * (1 - col("remise_pct") / 100)).cast(IntegerType())
    )
    .withColumn("frais_livraison", col("frais_livraison").cast(IntegerType()))
    .withColumn("date_commande_date", substring("date_commande", 1, 10))
)

display(df_clean)
df_clean.printSchema()

# Sauvegarder dans Silver
df_clean.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://WS_Projet_Ryma@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Tables/dbo/commandes_silver")

StatementMeta(, 00575fdf-f559-403e-9c2b-c739a7140374, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb4ddf23-2fe1-4264-978d-c757cecef379)

root
 |-- commande_id: string (nullable = true)
 |-- ligne: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- produit_id: string (nullable = true)
 |-- magasin_id: string (nullable = true)
 |-- quantite: string (nullable = true)
 |-- prix_unitaire: string (nullable = true)
 |-- remise_pct: string (nullable = true)
 |-- montant_brut: string (nullable = true)
 |-- montant_net: string (nullable = true)
 |-- frais_livraison: string (nullable = true)
 |-- moyen_paiement: string (nullable = true)
 |-- statut_commande: string (nullable = true)
 |-- canal_vente: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 95819c3d-b163-45db-8be9-38ab879419fc)

root
 |-- commande_id: string (nullable = true)
 |-- ligne: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- produit_id: string (nullable = true)
 |-- magasin_id: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix_unitaire: integer (nullable = true)
 |-- remise_pct: integer (nullable = true)
 |-- montant_brut: integer (nullable = true)
 |-- montant_net: integer (nullable = true)
 |-- frais_livraison: integer (nullable = true)
 |-- moyen_paiement: string (nullable = true)
 |-- statut_commande: string (nullable = true)
 |-- canal_vente: string (nullable = true)
 |-- date_commande_date: string (nullable = true)

